# 📘 Notebook 05 — Evaluation & Reporting

**Goals:**
1. Load both base and fine-tuned models
2. Run ROUGE + BERTScore evaluation (100 samples)
3. Run LLM-as-Judge evaluation (30 samples)
4. Side-by-side comparison
5. Export full evaluation report to CSV

**Runtime:** ~1 hr | **GPU:** Required | **Cost:** ~$0.05 (DeepSeek Chat)

## 1. Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import (
    SFT_LORA_SAVE_DIR, ILDC_DATASET,
    EVAL_N_SAMPLES, JUDGE_N_SAMPLES, EVAL_REPORT_PATH,
)

print(f"LoRA path: {SFT_LORA_SAVE_DIR}")
print(f"ROUGE/BERT samples: {EVAL_N_SAMPLES}")
print(f"LLM Judge samples: {JUDGE_N_SAMPLES}")

## 2. Load Both Models

In [ ]:
from src.model_utils import load_base_model, load_finetuned_model

# Base model (before fine-tuning)
base_model, tokenizer = load_base_model()
print()

# Fine-tuned model (LoRA adapter)
finetuned_model, _ = load_finetuned_model(SFT_LORA_SAVE_DIR)

## 3. Load Test Dataset

In [ ]:
from datasets import load_dataset

ildc_test = load_dataset(ILDC_DATASET, split="test")
print(f"Test cases: {len(ildc_test)}")

## 4. Quick Sanity Check

In [ ]:
from src.model_utils import generate_judgment

sample = ildc_test[0]
print("=== FACTS ===")
print(sample['text'][:500])
print("\n=== FINE-TUNED MODEL OUTPUT ===")
print(generate_judgment(finetuned_model, tokenizer, sample['text']))

## 5. ROUGE + BERTScore Evaluation (100 samples)

> Compares base model vs fine-tuned model on automated metrics.

In [ ]:
from src.evaluation import evaluate_rouge_bert, print_comparison_table

print("Evaluating BASE model...")
base_scores = evaluate_rouge_bert(base_model, tokenizer, ildc_test, n_samples=EVAL_N_SAMPLES)

print("\nEvaluating FINE-TUNED model...")
ft_scores = evaluate_rouge_bert(finetuned_model, tokenizer, ildc_test, n_samples=EVAL_N_SAMPLES)

print_comparison_table(base_scores, ft_scores)

## 6. LLM-as-Judge Evaluation (30 samples)

> DeepSeek-Chat scores each prediction on:
> - Legal Accuracy
> - Reasoning Chain
> - Reference Fidelity
> - Completeness
> 
> **Cost:** ~$0.05

In [ ]:
from src.evaluation import run_llm_judge_eval, print_judge_scores

print("Running LLM Judge on fine-tuned model (30 cases)...")
judge_results = run_llm_judge_eval(
    finetuned_model, tokenizer, ildc_test,
    n_samples=JUDGE_N_SAMPLES,
)

print_judge_scores(judge_results)

## 7. Side-by-Side Manual Analysis

In [ ]:
from src.evaluation import print_side_by_side

# Inspect a few cases manually
for idx in [0, 5, 12, 20]:
    print_side_by_side(base_model, finetuned_model, tokenizer, ildc_test, idx)
    print()

## 8. Export Full Evaluation Report

In [ ]:
from src.evaluation import export_report

df = export_report(judge_results, save_path=EVAL_REPORT_PATH)

print(f"\nTop 5 best cases:")
if 'overall' in df.columns:
    print(df.nlargest(5, 'overall')[['case_id', 'overall', 'feedback']])
    print(f"\nBottom 5 cases:")
    print(df.nsmallest(5, 'overall')[['case_id', 'overall', 'feedback']])

## 9. Summary

✅ **Completed:**
- ROUGE + BERTScore evaluation (base vs fine-tuned)
- LLM-as-Judge scoring (30 cases)
- Side-by-side manual comparison
- Full evaluation report exported to CSV

### Expected Targets:
| Metric | Target |
|--------|--------|
| ROUGE-L | > 0.30 |
| BERTScore F1 | > 0.85 |
| LLM Judge Overall | > 7/10 |

🎉 **Pipeline complete!**